# ML_Code_1 — Google Colab GPU
Run cells in order in a **GPU runtime**. This uses the complete original dataset, one GPU, and a fresh Colab work directory. Your SageMaker run is unaffected.

A faster GPU does not accelerate serial cleaning/SQLite/lexical features. Full-data completion within a Colab session is not guaranteed. Drive backups below contain portable checkpoints, logs and outputs, **not the large database/vector work state needed to resume after VM deletion**. Do not stop a progressing SageMaker run just to open this notebook.


In [ ]:
!nvidia-smi
!free -h
!df -h /content


## 1. Get the repository
Public repository: leave METHOD as github. For a private repository, download its ZIP from GitHub while signed in, upload that code ZIP to Colab's Files panel as `/content/ML_Code_1.zip`, and choose zip. Never paste a GitHub token into notebook code.


In [ ]:
from pathlib import Path
import subprocess, zipfile, shutil
METHOD = "github"  # github or zip
REPO = Path("/content/ML_Code_1")
if not REPO.exists():
    if METHOD == "github":
        result = subprocess.run(["git", "clone", "https://github.com/raoankit72005/ML_Code_1.git", str(REPO)], capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("Clone failed. If private, use the code-ZIP method described above. " + result.stderr)
    elif METHOD == "zip":
        unpack = Path("/content/ml_er_code_unpack")
        unpack.mkdir(exist_ok=True)
        with zipfile.ZipFile("/content/ML_Code_1.zip") as archive:
            for item in archive.infolist():
                target = (unpack / item.filename).resolve()
                if unpack.resolve() not in target.parents and target != unpack.resolve():
                    raise ValueError("Unsafe ZIP path")
            archive.extractall(unpack)
        matches = list(unpack.glob("*/hybrid.py"))
        if len(matches) != 1:
            raise ValueError("Expected one GitHub repository folder")
        shutil.move(str(matches[0].parent), str(REPO))
    else:
        raise ValueError("Choose github or zip")
assert (REPO / "colab/run.py").is_file(), "Repository predates Colab support"
print("Repository ready:", REPO)


## 2. Select training options and install
`lora` fine-tunes the shared encoder on training clusters. `frozen` skips fine-tuning to save time but still embeds every record; it may reduce matching quality. Both train LightGBM.

`cuda` compiles GPU LightGBM using this GPU's compute capability and tests it. Compilation needs nvcc/g++; failure is reported, never silently switched to CPU. Choose `cpu` explicitly to avoid that build and run only the encoder on GPU.

Packages go into a dedicated `/content/ml_er_env`; Colab's notebook Python is not replaced. Installation requires internet, disk space and time. Rerun setup after VM deletion, not during training.


In [ ]:
ENCODER_MODE = "lora"  # lora or frozen
MATCHER = "cuda"       # cuda or cpu
subprocess.run(["bash", str(REPO / "colab/setup.sh"), MATCHER], check=True)
PYTHON = "/content/ml_er_env/bin/python"


## 3. Mount Drive and copy the original dataset locally
Upload the original `ML_dataset.zip` to your own Google Drive first, or change DRIVE_ZIP below. Do not publish the challenge data. Your S3 URI is not a local filename and requires an authenticated transfer before this step.

Use a new RUN_NAME for a new experiment. Keep the same name only while resuming the same preserved Colab runtime/configuration. The copy is streamed to disk, not loaded into RAM.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ZIP = Path("/content/drive/MyDrive/ML_dataset.zip")
RUN_NAME = "ml_code_1_colab_01"
assert RUN_NAME and all(c.isalnum() or c in "_-" for c in RUN_NAME)
WORK = Path("/content") / RUN_NAME
BACKUP = Path("/content/drive/MyDrive/ML_Code_1_backups") / RUN_NAME
LOCAL_ZIP = Path("/content/ML_dataset.zip")
if not DRIVE_ZIP.is_file():
    raise FileNotFoundError(f"Upload the original dataset or correct DRIVE_ZIP: {DRIVE_ZIP}")
identity = lambda p: (p.stat().st_size, p.stat().st_mtime_ns)
if not LOCAL_ZIP.exists() or identity(LOCAL_ZIP) != identity(DRIVE_ZIP):
    if (WORK / "hybrid_manifest.json").exists():
        raise RuntimeError("Input changed during an existing run. Preserve it and choose a new experiment.")
    if shutil.disk_usage("/content").free < DRIVE_ZIP.stat().st_size + 20 * 1024**3:
        raise RuntimeError("Insufficient local disk even for startup. Drive capacity does not increase local disk.")
    temporary = LOCAL_ZIP.with_suffix(".copying")
    shutil.copy2(DRIVE_ZIP, temporary)
    temporary.replace(LOCAL_ZIP)
assert zipfile.is_zipfile(LOCAL_ZIP), "Input is not a ZIP"
print("Local dataset:", LOCAL_ZIP, "bytes:", LOCAL_ZIP.stat().st_size)


## 4. Inspect detected resources and generated settings
This is a minimum startup check, not proof that every full-data stage fits. Later stages estimate RAM/VRAM/disk needs and stop if unsafe. No training rows or test queries are silently sampled. Settings are saved once for reproducibility.


In [ ]:
COMMAND = [PYTHON, "-u", str(REPO / "colab/run.py"),
           "--input", str(LOCAL_ZIP), "--work", str(WORK), "--backup", str(BACKUP),
           "--encoder-mode", ENCODER_MODE, "--matcher", MATCHER]
subprocess.run(COMMAND + ["--check-only"], check=True)


## 5. Run cleaning → matching results
Output prints stage starts, encoder loss and throughput, sampled recall, and LightGBM metrics. Backups occur approximately every three minutes and at exit. Completed stages can resume when the same local work files remain. This launcher does not automatically restore an old runtime from Drive checkpoints.

Do not edit configs, pull new code, run setup or start a second copy during training. If a resource guard rejects the job, preserve its output and adjust a new experiment deliberately. Changing to a CPU matcher can help VRAM but does not remove RAM or disk requirements.


In [ ]:
# Stream subprocess output to the notebook. An interrupt is forwarded to the launcher.
import signal
job = subprocess.Popen(COMMAND, cwd=REPO)
try:
    status = job.wait()
except KeyboardInterrupt:
    job.send_signal(signal.SIGINT)
    print("Stop requested. Wait for the launcher to stop its pipeline and back up artifacts.")
    raise
if status:
    raise RuntimeError(f"Pipeline exited {status}; inspect the last traceback/resource plan above.")


## 6. Inspect final outputs
This cell checks completion; it does not treat partial files as a valid submission. Run the competition-provided validator too. Your local full-ID audit is copied to Drive.


In [ ]:
import json
state = json.loads((WORK / "hybrid_manifest.json").read_text())
assert "audit" in state["completed"], "Pipeline has not completed the final audit"
for name in ("matching_results.tsv", "candidate_pairs.tsv", "submission_coverage.json"):
    path = BACKUP / "test" / name
    assert path.exists(), f"Missing backup artifact: {path}"
    print(path, path.stat().st_size, "bytes")
print((BACKUP / "test/submission_coverage.json").read_text())


## Optional: TensorBoard after or alongside training
Do not launch another pipeline to monitor it. For a separate monitoring session, use the saved JSONL files or TensorBoard events under WORK/logs/tensorboard. Portable JSONL logs are also backed up to Drive.

Colab usage and VM lifetimes: https://research.google.com/colaboratory/faq.html
LightGBM CUDA build: https://lightgbm.readthedocs.io/en/v4.6.0/Installation-Guide.html
